# Fine-Tuned BERT for Enzyme Classification — ICAICTA 2024

This notebook is a curated extraction of the original 2024 research notebook.

**Output provenance.** Visible model-training and evaluation outputs in this notebook are retained from the original historical execution. During the September 2026 cleanup, the current execution environment could not install/download Hugging Face `transformers` models and does not provide the original CUDA setup, so the full BERT fine-tuning run could not be genuinely re-executed here.

The cleanup does, however, fix state/path issues discovered in the legacy notebook and validates the bundled tabular datasets.


## Fine Tuning

In [ ]:
import numpy as np
import pandas as pd

import itertools
from tqdm import tqdm
import random
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score, roc_auc_score, classification_report, confusion_matrix
from sklearn.metrics import adjusted_rand_score, adjusted_mutual_info_score, fowlkes_mallows_score, v_measure_score, homogeneity_score, completeness_score

import torch
from torch import optim
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F

from transformers import BertForSequenceClassification, BertConfig, BertTokenizer

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
bp_iea = pd.read_excel('../data/bp_iea_gene_class_def_clean.xlsx')
bp_non_iea = pd.read_excel('../data/bp_non_iea_gene_class_def_clean.xlsx')
mf_iea = pd.read_excel('../data/mf_iea_gene_class_def_clean.xlsx')
mf_non_iea = pd.read_excel('../data/mf_non_iea_gene_class_def_clean.xlsx')
cc_iea = pd.read_excel('../data/cc_iea_gene_class_def_clean.xlsx')
cc_non_iea = pd.read_excel('../data/cc_non_iea_gene_class_def_clean.xlsx')

In [ ]:
bp_iea['class'].value_counts()

class
2    4401
3    2887
6    1133
1     864
5     645
4     587
Name: count, dtype: int64

In [ ]:
bp_non_iea['class'].value_counts()

class
2    1405
3     978
6     212
1     192
4     151
5     149
Name: count, dtype: int64

In [ ]:
class DocumentSentimentDataset(Dataset):
    # Static constant variable
    LABEL2INDEX = {1: 0, 2: 1, 3: 2, 4: 3, 5: 4, 6: 5}
    INDEX2LABEL = {0: 1, 1: 2, 2: 3, 3: 4, 4: 5, 5: 6}
    NUM_LABELS = 6

    def load_dataset(self, path):
        df = path
        df.columns = ['gene', 'def_clean', 'class']
        df['class'] = df['class'].apply(lambda lab: self.LABEL2INDEX[lab])
        return df

    def __init__(self, dataset_path, tokenizer, no_special_token=False, *args, **kwargs):
        self.data = self.load_dataset(dataset_path)
        self.tokenizer = tokenizer
        self.no_special_token = no_special_token

    def __getitem__(self, index):
        data = self.data.loc[index,:]
        text, sentiment = data['def_clean'], data['class']
        subwords = self.tokenizer.encode(text, add_special_tokens=not self.no_special_token)
        return np.array(subwords), np.array(sentiment), data['def_clean']

    def __len__(self):
        return len(self.data)

class DocumentSentimentDataLoader(DataLoader):
    def __init__(self, max_seq_len=512, *args, **kwargs):
        super(DocumentSentimentDataLoader, self).__init__(*args, **kwargs)
        self.collate_fn = self._collate_fn
        self.max_seq_len = max_seq_len

    def _collate_fn(self, batch):
        batch_size = len(batch)
        max_seq_len = max(map(lambda x: len(x[0]), batch))
        max_seq_len = min(self.max_seq_len, max_seq_len)

        subword_batch = np.zeros((batch_size, max_seq_len), dtype=np.int64)
        mask_batch = np.zeros((batch_size, max_seq_len), dtype=np.float32)
        sentiment_batch = np.zeros((batch_size, 1), dtype=np.int64)

        seq_list = []
        for i, (subwords, sentiment, raw_seq) in enumerate(batch):
            subwords = subwords[:max_seq_len]
            subword_batch[i,:len(subwords)] = subwords
            mask_batch[i,:len(subwords)] = 1
            sentiment_batch[i,0] = sentiment

            seq_list.append(raw_seq)

        return subword_batch, mask_batch, sentiment_batch, seq_list

In [ ]:
def get_lr(optimizer):
    for param_group in optimizer.param_groups:
        return param_group['lr']

def count_param(module, trainable=False):
    if trainable:
        return sum(p.numel() for p in module.parameters() if p.requires_grad)
    else:
        return sum(p.numel() for p in module.parameters())

def metrics_to_string(metric_dict):
    string_list = []
    for key, value in metric_dict.items():
        string_list.append('{}:{:.2f}'.format(key, value))
    return ' '.join(string_list)

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)

set_seed(26092020)

In [ ]:
def forward_sequence_classification(model, batch_data, i2w, is_test=False, device='cpu', **kwargs):
    # Unpack batch data
    if len(batch_data) == 3:
        (subword_batch, mask_batch, label_batch) = batch_data
        token_type_batch = None
    elif len(batch_data) == 4:
        (subword_batch, mask_batch, token_type_batch, label_batch) = batch_data

    # Prepare input & label
    subword_batch = torch.LongTensor(subword_batch)
    mask_batch = torch.FloatTensor(mask_batch)
    token_type_batch = torch.LongTensor(token_type_batch) if token_type_batch is not None else None
    label_batch = torch.LongTensor(label_batch)

    if device == "cuda":
        subword_batch = subword_batch.cuda()
        mask_batch = mask_batch.cuda()
        token_type_batch = token_type_batch.cuda() if token_type_batch is not None else None
        label_batch = label_batch.cuda()

    # Forward model
    outputs = model(subword_batch, attention_mask=mask_batch, token_type_ids=token_type_batch, labels=label_batch)
    loss, logits = outputs[:2]

    # generate prediction & label list
    list_hyp = []
    list_label = []
    hyp = torch.topk(logits, 1)[1]
    for j in range(len(hyp)):
        list_hyp.append(i2w[hyp[j].item()])
        list_label.append(i2w[label_batch[j][0].item()])

    return loss, list_hyp, list_label

In [ ]:
def document_sentiment_metrics_fn(list_hyp, list_label):
    metrics = {}
    metrics["F1"] = f1_score(list_label, list_hyp, average='macro')
    metrics["ARI"] = adjusted_rand_score(list_label, list_hyp)
    metrics["AMI"] = adjusted_mutual_info_score(list_label, list_hyp)
    metrics["FM Score"] = fowlkes_mallows_score(list_label, list_hyp)
    metrics["Homogeneity Score"] = homogeneity_score(list_label, list_hyp)
    metrics["Completeness Score"] = completeness_score(list_label, list_hyp)
    metrics["V Measure"] = v_measure_score(list_label, list_hyp)
    return metrics

In [ ]:
# Load model directly
from transformers import BertConfig, AutoTokenizer, AutoModelForSequenceClassification

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
config = BertConfig.from_pretrained("bert-base-uncased", num_labels = 6)
model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", config = config)

In [ ]:
bp_non_iea_test = pd.read_excel('../data/bp_non_iea_test.xlsx')
bp_non_iea_train = bp_non_iea.loc[
    ~bp_non_iea['gene'].isin(bp_non_iea_test['gene'])
].reset_index(drop=True)

print(f"Train rows: {len(bp_non_iea_train):,}")
print(f"Test rows:  {len(bp_non_iea_test):,}")


Train rows: 2,160
Test rows:  927


In [ ]:
from sklearn.model_selection import train_test_split

X = cc_iea[['gene', 'def_clean']]
y = cc_iea['class']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.3, stratify = y)

In [ ]:
X_train

,gene,def_clean
0,"(Gene(GeneName='P55025', GOAnnotations=['GO:00...",accumulation pigment organism tissue cell eith...
1,"(Gene(GeneName='O22229', GOAnnotations=['GO:00...",process maintains redox environment cell compa...
2,"(Gene(GeneName='P61076', GOAnnotations=['GO:00...",process maintains redox environment cell compa...
3,"(Gene(GeneName='O23207', GOAnnotations=['GO:00...",metabolic process results removal addition one...
4,"(Gene(GeneName='P74312', GOAnnotations=['GO:00...",process results change state activity cell org...
...,...,...
2155,"(Gene(GeneName='P53630', GOAnnotations=['GO:00...",chemical reactions pathways resulting formatio...
2156,"(Gene(GeneName='Q55849', GOAnnotations=['GO:00...",chemical reactions pathways resulting formatio...
2157,"(Gene(GeneName='Q7NFL5', GOAnnotations=['GO:00...",chemical reactions pathways resulting formatio...
2158,"(Gene(GeneName='Q7URG0', GOAnnotations=['GO:00...",chemical reactions pathways resulting formatio...


In [ ]:
X_train = X_train.reset_index(drop = True)
X_test = X_test.reset_index(drop = True)
y_train = y_train.reset_index(drop = True)
y_test = y_test.reset_index(drop = True)

In [ ]:
train = pd.concat([X_train, y_train], axis = 1)

In [ ]:
train['class'].value_counts()

class
2    983
3    684
6    148
1    135
4    106
5    104
Name: count, dtype: int64

In [ ]:
train

,gene,def_clean,class
0,"(Gene(GeneName='P55025', GOAnnotations=['GO:00...",accumulation pigment organism tissue cell eith...,1
1,"(Gene(GeneName='O22229', GOAnnotations=['GO:00...",process maintains redox environment cell compa...,1
2,"(Gene(GeneName='P61076', GOAnnotations=['GO:00...",process maintains redox environment cell compa...,1
3,"(Gene(GeneName='O23207', GOAnnotations=['GO:00...",metabolic process results removal addition one...,1
4,"(Gene(GeneName='P74312', GOAnnotations=['GO:00...",process results change state activity cell org...,1
...,...,...,...
2155,"(Gene(GeneName='P53630', GOAnnotations=['GO:00...",chemical reactions pathways resulting formatio...,6
2156,"(Gene(GeneName='Q55849', GOAnnotations=['GO:00...",chemical reactions pathways resulting formatio...,6
2157,"(Gene(GeneName='Q7NFL5', GOAnnotations=['GO:00...",chemical reactions pathways resulting formatio...,6
2158,"(Gene(GeneName='Q7URG0', GOAnnotations=['GO:00...",chemical reactions pathways resulting formatio...,6


In [ ]:
y_test.value_counts()

class
2    422
3    294
6     64
1     57
5     45
4     45
Name: count, dtype: int64

In [ ]:
dataset = DocumentSentimentDataset(train, tokenizer)
data_loader = DocumentSentimentDataLoader(dataset=dataset, max_seq_len = 512, batch_size = 8, num_workers = 2, shuffle=True)

In [ ]:
w2i, i2w = DocumentSentimentDataset.LABEL2INDEX, DocumentSentimentDataset.INDEX2LABEL

In [ ]:
torch.cuda.is_available()

True

In [ ]:
optimizer = optim.Adam(model.parameters(), lr=3e-6)
model = model.cuda()

n_epochs = 10
for epoch in range(n_epochs):
    model.train()
    torch.set_grad_enabled(True)

    total_train_loss = 0
    list_hyp, list_label = [], []

    train_pbar = tqdm(data_loader, leave=True, total=len(data_loader))
    for i, batch_data in enumerate(train_pbar):
        # Forward model
        loss, batch_hyp, batch_label = forward_sequence_classification(model, batch_data[:-1], i2w=i2w, device = 'cuda')

        # Update model
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        tr_loss = loss.item()
        total_train_loss = total_train_loss + tr_loss

        # Calculate metrics
        list_hyp += batch_hyp
        list_label += batch_label

        train_pbar.set_description("(Epoch {}) TRAIN LOSS:{:.4f} LR:{:.8f}".format((epoch+1),
            total_train_loss/(i+1), get_lr(optimizer)))

    # Calculate train metric
    metrics = document_sentiment_metrics_fn(list_hyp, list_label)
    print("(Epoch {}) TRAIN LOSS:{:.4f} {} LR:{:.8f}".format((epoch+1),
        total_train_loss/(i+1), metrics_to_string(metrics), get_lr(optimizer)))

(Epoch 1) TRAIN LOSS:1.4157 LR:0.00000300: 100%|██████████| 270/270 [02:07<00:00,  2.12it/s]


(Epoch 1) TRAIN LOSS:1.4157 F1:0.14 ARI:0.01 AMI:0.01 FM Score:0.52 Homogeneity Score:0.01 Completeness Score:0.05 V Measure:0.02 LR:0.00000300


(Epoch 2) TRAIN LOSS:1.1688 LR:0.00000300: 100%|██████████| 270/270 [02:03<00:00,  2.18it/s]


(Epoch 2) TRAIN LOSS:1.1688 F1:0.32 ARI:0.15 AMI:0.15 FM Score:0.52 Homogeneity Score:0.12 Completeness Score:0.22 V Measure:0.15 LR:0.00000300


(Epoch 3) TRAIN LOSS:0.9458 LR:0.00000300: 100%|██████████| 270/270 [02:02<00:00,  2.20it/s]


...

(Epoch 8) TRAIN LOSS:0.4141 F1:0.80 ARI:0.76 AMI:0.65 FM Score:0.84 Homogeneity Score:0.63 Completeness Score:0.67 V Measure:0.65 LR:0.00000300


  0%|          | 0/270 [00:00<?, ?it/s]Token indices sequence length is longer than the specified maximum sequence length for this model (1320 > 512). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (524 > 512). Running this sequence through the model will result in indexing errors
(Epoch 9) TRAIN LOSS:0.3491 LR:0.00000300: 100%|██████████| 270/270 [02:06<00:00,  2.13it/s]


(Epoch 9) TRAIN LOSS:0.3491 F1:0.84 ARI:0.79 AMI:0.69 FM Score:0.86 Homogeneity Score:0.67 Completeness Score:0.70 V Measure:0.69 LR:0.00000300


(Epoch 10) TRAIN LOSS:0.3083 LR:0.00000300: 100%|██████████| 270/270 [02:03<00:00,  2.18it/s]

(Epoch 10) TRAIN LOSS:0.3083 F1:0.86 ARI:0.81 AMI:0.71 FM Score:0.87 Homogeneity Score:0.71 Completeness Score:0.73 V Measure:0.72 LR:0.00000300


In [ ]:
test = pd.concat([X_test, y_test], axis = 1)

In [ ]:
def predict(text):
  sentiments = []
  scores = []

  with torch.no_grad():
    for index in tqdm(text.index):
      subwords = tokenizer.encode(text[index], padding = True, truncation = True, max_length = 512, add_special_tokens = True)
      subwords = torch.LongTensor(subwords).view(1, -1).to(model.device)

      logits = model(subwords)[0]
      label = torch.topk(logits, k=1, dim=-1)[1].squeeze().item()

      sentiments.append(i2w[label])
      scores.append(F.softmax(logits, dim=-1).squeeze()[label])

    return sentiments, scores

sentimen, skor = predict(test['def_clean'])

100%|██████████| 927/927 [00:12<00:00, 75.67it/s]


In [ ]:
from sklearn.metrics import adjusted_rand_score, fowlkes_mallows_score, v_measure_score, homogeneity_score, completeness_score

print("ARI: ", adjusted_rand_score(sentimen, test['class']))
print("AMI: ", adjusted_mutual_info_score(sentimen, test['class']))
print("FM Score: ", fowlkes_mallows_score(sentimen, test['class']))
print("Homogeneity: ", homogeneity_score(sentimen, test['class']))
print("Completeness: ", completeness_score(sentimen, test['class']))
print("V-Measure: ", v_measure_score(sentimen, test['class']))
print("Accuracy score: ", accuracy_score(sentimen, test['class']))
print("F1 Score: ", f1_score(sentimen, test['class'], average = 'macro'))

ARI:  0.5767535394098131
AMI:  0.5268798113896077
FM Score:  0.7219033108182388
Homogeneity:  0.5473972736762498
Completeness:  0.5174635429172827
V-Measure:  0.5320096827788452
Accuracy score:  0.81445523193096
F1 Score:  0.7598527968014311


**BP IEA**
-  ARI:  0.8846167684293891
- FM Score:  0.9159944728596471
- Homogeneity:  0.8174438499099075
- Completeness:  0.8283334700910963
- V-Measure:  0.8228526333077502
- Accuracy score:  0.9426489226869454
- F1 Score:  0.9110649024511996
- Inference time: 40 second
- finetuning time: 2035 second
- epoch: 5

**BP NONIEA**
- ARI:  0.45838613800995875
- AMI:  0.3925163772882105
- FM Score:  0.6625263837255447
- Homogeneity:  0.4569219854120013
- Completeness:  0.3553284662844516
- V-Measure:  0.3997717402286352
- Accuracy score:  0.7551240560949298
- F1 Score:  0.5805563176957925
- Inference time: 11 second
- finetuning time: 600 second
- epoch: 5

**MF IEA**
- ARI:  0.9812709847107433
- AMI:  0.9683045867587164
- FM Score:  0.9868423128968399
- Homogeneity:  0.9683966963467856
- Completeness:  0.9683933290884897
- V-Measure:  0.9683950127147105
- Accuracy score:  0.9925154572079401
- F1 Score:  0.9909231287894416
- Inference time: 69 second
- finetuning time: 742 second
- epoch: 2

**MF NONIEA**
- ARI:  0.858024434461084
- AMI:  0.7885466425276002
- FM Score:  0.90354186411643
- Homogeneity:  0.8015253484812535
- Completeness:  0.7799381805512727
- V-Measure:  0.7905844307932015
- Accuracy score:  0.9319526627218935
- F1 Score:  0.8791689492696233
- Inference time: 22 second
- finetuning time: 380 second
- epoch: 5

**CC IEA**
- ARI:  0.17236306068378657
- AMI:  0.1715904885479089
- FM Score:  0.46821648426244605
- Homogeneity:  0.2159399832433634
- Completeness:  0.14787885332587822
- V-Measure:  0.1755431764356186
- Accuracy score:  0.563006300630063
- F1 Score:  0.38279544143669547
- inference time: 9 second
- finetuning time: 1960 second
- epoch: 10

**CC NONIEA**
- ARI:  0.24481804009664265
- AMI:  0.195288457050826
- FM Score:  0.5354933704478795
- Homogeneity:  0.25105149305059865
- Completeness:  0.17422797434110407
- V-Measure:  0.20570093993853109
- Accuracy score:  0.6442080378250591
- F1 Score:  0.4153765168224253
- inference time: 10 second
- finetuning time: 810 second
- epoch: 10

# Subclass Finetuning

In [ ]:
import numpy as np
import pandas as pd

import itertools
from tqdm import tqdm
import random
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score, roc_auc_score, classification_report, confusion_matrix
from sklearn.metrics import adjusted_rand_score, adjusted_mutual_info_score, fowlkes_mallows_score, v_measure_score, homogeneity_score, completeness_score

import torch
from torch import optim
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F

from transformers import BertForSequenceClassification, BertConfig, BertTokenizer

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
bp_iea = pd.read_excel('../data/bp_iea_subclass.xlsx')
bp_non_iea = pd.read_excel('../data/bp_noniea_subclass.xlsx')
mf_iea = pd.read_excel('../data/mf_iea_subclass.xlsx')
mf_non_iea = pd.read_excel('../data/mf_noniea_subclass.xlsx')
cc_iea = pd.read_excel('../data/cc_iea_subclass.xlsx')
cc_non_iea = pd.read_excel('../data/cc_noniea_subclass.xlsx')

In [ ]:
bp_iea.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10614 entries, 0 to 10613
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   gene       10614 non-null  object 
 1   clusters   10614 non-null  float64
 2   def_clean  10614 non-null  object 
dtypes: float64(1), object(2)
memory usage: 248.9+ KB


In [ ]:
bp_iea['clusters'].unique()

array([6.1 , 3.2 , 3.1 , 2.7 , 1.14, 4.2 , 3.4 , 6.3 , 3.6 , 1.18, 1.8 ,
       2.5 , 4.1 , 1.6 , 1.12, 2.3 , 6.5 , 1.5 , 4.6 , 2.1 , 5.6 , 1.17,
       3.5 , 2.4 , 4.3 , 1.9 , 5.2 , 1.11, 6.2 , 1.3 , 1.1 , 2.2 , 2.8 ,
       5.3 , 5.1 , 5.4 , 1.4 , 1.2 , 4.99, 1.16, 2.6 , 4.4 , 1.15, 1.13])

In [ ]:
bp_iea['clusters'].nunique()

44

In [ ]:
bp_iea['clusters'].value_counts()

clusters
2.70    1891
3.10    1213
2.30     905
2.10     792
3.60     627
6.10     570
6.30     446
3.20     420
3.40     360
2.50     313
2.40     273
3.50     267
4.10     239
4.20     236
5.20     220
1.10     215
5.30     176
5.40     161
2.80     118
1.20      98
1.17      94
6.50      91
5.10      88
5.60      71
1.30      70
2.60      55
1.50      55
2.20      54
4.30      52
1.90      51
1.80      51
1.11      50
4.60      49
1.18      43
1.14      39
1.15      27
4.99      26
6.20      26
1.40      23
1.16      14
1.60      12
1.12      12
4.40      11
1.13      10
Name: count, dtype: int64

In [ ]:
cc_non_iea[cc_non_iea['clusters'] == 1.13]

,gene,clusters,def_clean
2700,Q5M827_CC,1.13,contents cell excluding plasma membrane nucleu...


In [ ]:
mf_non_iea['clusters'].value_counts()

In [ ]:
mf_iea = mf_iea.drop(mf_iea[(mf_iea['clusters'] == 7.60) | (mf_iea['clusters'] == 7.30) | (mf_iea['clusters'] == 7.10)].index)

In [ ]:
mf_iea.shape


In [ ]:
mf_iea['def_clean'].isna().sum()

0

In [ ]:
mf_iea['clusters'].nunique()

47

In [ ]:
mf_iea['clusters'].unique()

array([6.1 , 3.2 , 3.1 , 2.7 , 1.14, 4.2 , 3.4 , 6.3 , 3.6 , 1.18, 1.8 ,
       2.5 , 4.1 , 1.6 , 1.12, 2.3 , 6.5 , 1.5 , 4.6 , 2.1 , 5.6 , 1.17,
       7.1 , 3.5 , 2.4 , 4.3 , 1.9 , 5.2 , 1.11, 6.2 , 1.3 , 1.1 , 2.2 ,
       2.8 , 5.3 , 5.1 , 5.4 , 1.4 , 1.2 , 4.99, 1.16, 7.3 , 2.6 , 4.4 ,
       1.15, 1.13, 7.6 ])

In [ ]:
class DocumentSentimentDataset(Dataset):
    # Static constant variable
    LABEL2INDEX = {'6.1' : 0, '3.2' : 1, '3.1' : 2, '2.7' : 3, '1.14': 4, '4.2' : 5, '3.4' : 6,
                   '6.3' : 7, '3.6' : 8, '1.18' : 9, '1.8' : 10, '2.5' : 11, '4.1' : 12, '1.6' : 13,
                   '2.3' : 15, '6.5' : 16, '1.5' : 17, '4.6' : 18, '2.1' : 19,
                   '5.6' : 20, '1.17': 21, '3.5' : 22, '2.4' : 23, '4.3' : 24, '1.9' : 25, '5.2' : 26,
                   '1.11': 27, '6.2' : 28, '1.3' : 29, '1.1' : 30, '2.2' : 31, '2.8' : 32,
                   '5.3' : 33, '5.1' : 34, '5.4' : 35, '1.4' : 36, '1.2' : 37, '4.99': 38, '1.16': 39,
                   '2.6' : 40, '4.4' : 41, '1.15': 42, '1.13': 43}
    INDEX2LABEL = {0: '6.1' , 1: '3.2', 2: '3.1', 3: '2.7', 4: '1.14', 5: '4.2', 6: '3.4',
                   7: '6.3', 8: '3.6', 9: '1.18', 10: '1.8', 11: '2.5', 12: '4.1', 13: '1.6',
                   15: '2.3', 16: '6.5', 17: '1.5', 18: '4.6', 19: '2.1',
                   20: '5.6', 21: '1.17', 22: '3.5', 23: '2.4', 24: '4.3', 25: '1.9', 26: '5.2',
                   27: '1.11', 28: '6.2', 29: '1.3', 30: '1.11', 31: '2.2', 32: '2.8',
                   33: '5.3', 34: '5.1', 35: '5.4', 36: '1.4', 37: '1.2', 38: '4.99', 39: '1.16',
                   40: '2.6', 41: '4.4', 42: '1.15', 43: '1.13'}
    NUM_LABELS = 43

    def load_dataset(self, path):
        df = path
        df.columns = ['gene', 'def_clean', 'clusters']
        df['clusters'] = df['clusters'].apply(lambda lab: self.LABEL2INDEX[lab])
        return df

    def __init__(self, dataset_path, tokenizer, no_special_token=False, *args, **kwargs):
        self.data = self.load_dataset(dataset_path)
        self.tokenizer = tokenizer
        self.no_special_token = no_special_token

    def __getitem__(self, index):
        data = self.data.loc[index,:]
        text, sentiment = data['def_clean'], data['clusters']
        subwords = self.tokenizer.encode(text, add_special_tokens=not self.no_special_token)
        return np.array(subwords), np.array(sentiment), data['def_clean']

    def __len__(self):
        return len(self.data)

class DocumentSentimentDataLoader(DataLoader):
    def __init__(self, max_seq_len=512, *args, **kwargs):
        super(DocumentSentimentDataLoader, self).__init__(*args, **kwargs)
        self.collate_fn = self._collate_fn
        self.max_seq_len = max_seq_len

    def _collate_fn(self, batch):
        batch_size = len(batch)
        max_seq_len = max(map(lambda x: len(x[0]), batch))
        max_seq_len = min(self.max_seq_len, max_seq_len)

        subword_batch = np.zeros((batch_size, max_seq_len), dtype=np.int64)
        mask_batch = np.zeros((batch_size, max_seq_len), dtype=np.float32)
        sentiment_batch = np.zeros((batch_size, 1), dtype=np.int64)

        seq_list = []
        for i, (subwords, sentiment, raw_seq) in enumerate(batch):
            subwords = subwords[:max_seq_len]
            subword_batch[i,:len(subwords)] = subwords
            mask_batch[i,:len(subwords)] = 1
            sentiment_batch[i,0] = sentiment

            seq_list.append(raw_seq)

        return subword_batch, mask_batch, sentiment_batch, seq_list

In [ ]:
def get_lr(optimizer):
    for param_group in optimizer.param_groups:
        return param_group['lr']

def count_param(module, trainable=False):
    if trainable:
        return sum(p.numel() for p in module.parameters() if p.requires_grad)
    else:
        return sum(p.numel() for p in module.parameters())

def metrics_to_string(metric_dict):
    string_list = []
    for key, value in metric_dict.items():
        string_list.append('{}:{:.2f}'.format(key, value))
    return ' '.join(string_list)

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)

set_seed(26092020)

In [ ]:
def forward_sequence_classification(model, batch_data, i2w, is_test=False, device='cpu', **kwargs):
    # Unpack batch data
    if len(batch_data) == 3:
        (subword_batch, mask_batch, label_batch) = batch_data
        token_type_batch = None
    elif len(batch_data) == 4:
        (subword_batch, mask_batch, token_type_batch, label_batch) = batch_data

    # Prepare input & label
    subword_batch = torch.LongTensor(subword_batch)
    mask_batch = torch.FloatTensor(mask_batch)
    token_type_batch = torch.LongTensor(token_type_batch) if token_type_batch is not None else None
    label_batch = torch.LongTensor(label_batch)

    if device == "cuda":
        subword_batch = subword_batch.cuda()
        mask_batch = mask_batch.cuda()
        token_type_batch = token_type_batch.cuda() if token_type_batch is not None else None
        label_batch = label_batch.cuda()

    # Forward model
    outputs = model(subword_batch, attention_mask=mask_batch, token_type_ids=token_type_batch, labels=label_batch)
    loss, logits = outputs[:2]

    # generate prediction & label list
    list_hyp = []
    list_label = []
    hyp = torch.topk(logits, 1)[1]
    for j in range(len(hyp)):
        list_hyp.append(i2w[hyp[j].item()])
        list_label.append(i2w[label_batch[j][0].item()])

    return loss, list_hyp, list_label

In [ ]:
def document_sentiment_metrics_fn(list_hyp, list_label):
    metrics = {}
    metrics["F1"] = f1_score(list_label, list_hyp, average='macro')
    metrics["ARI"] = adjusted_rand_score(list_label, list_hyp)
    metrics["AMI"] = adjusted_mutual_info_score(list_label, list_hyp)
    metrics["FM Score"] = fowlkes_mallows_score(list_label, list_hyp)
    metrics["Homogeneity Score"] = homogeneity_score(list_label, list_hyp)
    metrics["Completeness Score"] = completeness_score(list_label, list_hyp)
    metrics["V Measure"] = v_measure_score(list_label, list_hyp)
    return metrics

In [ ]:
# Load model directly
from transformers import BertConfig, AutoTokenizer, AutoModelForSequenceClassification

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
config = BertConfig.from_pretrained("bert-base-uncased", num_labels = 44)
model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", config = config)

/usr/local/lib/python3.10/dist-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
from sklearn.model_selection import train_test_split

X = bp_iea[['gene', 'def_clean']]
y = bp_iea['clusters']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.3, stratify = y)

In [ ]:
X_train = X_train.reset_index(drop = True)
y_train = y_train.reset_index(drop = True)
X_test = X_test.reset_index(drop = True)
y_test = y_test.reset_index(drop = True)

In [ ]:
train = pd.concat([X_train, y_train], axis = 1)

In [ ]:
train

,gene,def_clean,clusters
0,B5ZCB6_BP_I,posttranscriptional addition methyl groups spe...,2.1
1,A8L3U5_BP_I,synthesis aminoacyl tRNA formation ester bond ...,6.1
2,Q7NFK0_BP_I,chemical reactions pathways resulting formatio...,2.7
3,Q5SK89_BP_I,chemical reactions pathways resulting formatio...,2.3
4,A6VKH3_BP_I,chemical reactions pathways involving glycerol...,2.8
...,...,...,...
7424,Q44056_BP_I,chemical reactions pathways resulting breakdow...,3.5
7425,P36693_BP_I,cellular metabolic process involving deoxyribo...,3.1
7426,B0B9W8_BP_I,process removing one phosphoric ester anhydrid...,3.1
7427,P14924_BP_I,process removing one phosphoric ester anhydrid...,3.1


In [ ]:
train['clusters'] = train['clusters'].apply(lambda x: str(x))

In [ ]:
dataset = DocumentSentimentDataset(train, tokenizer)
data_loader = DocumentSentimentDataLoader(dataset=dataset, max_seq_len = 512, batch_size = 8, num_workers = 2, shuffle=True)

In [ ]:
w2i, i2w = DocumentSentimentDataset.LABEL2INDEX, DocumentSentimentDataset.INDEX2LABEL

In [ ]:
optimizer = optim.Adam(model.parameters(), lr=3e-6)
model = model.cuda()

n_epochs = 5
for epoch in range(n_epochs):
    model.train()
    torch.set_grad_enabled(True)

    total_train_loss = 0
    list_hyp, list_label = [], []

    train_pbar = tqdm(data_loader, leave=True, total=len(data_loader))
    for i, batch_data in enumerate(train_pbar):
        # Forward model
        loss, batch_hyp, batch_label = forward_sequence_classification(model, batch_data[:-1], i2w=i2w, device = 'cuda')

        # Update model
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        tr_loss = loss.item()
        total_train_loss = total_train_loss + tr_loss

        # Calculate metrics
        list_hyp += batch_hyp
        list_label += batch_label

        train_pbar.set_description("(Epoch {}) TRAIN LOSS:{:.4f} LR:{:.8f}".format((epoch+1),
            total_train_loss/(i+1), get_lr(optimizer)))

    # Calculate train metric
    metrics = document_sentiment_metrics_fn(list_hyp, list_label)
    print("(Epoch {}) TRAIN LOSS:{:.4f} {} LR:{:.8f}".format((epoch+1),
        total_train_loss/(i+1), metrics_to_string(metrics), get_lr(optimizer)))

(Epoch 1) TRAIN LOSS:0.6308 LR:0.00000300: 100%|██████████| 929/929 [06:50<00:00,  2.26it/s]


(Epoch 1) TRAIN LOSS:0.6308 F1:0.55 ARI:0.88 AMI:0.85 FM Score:0.89 Homogeneity Score:0.82 Completeness Score:0.88 V Measure:0.85 LR:0.00000300


(Epoch 2) TRAIN LOSS:0.5188 LR:0.00000300: 100%|██████████| 929/929 [06:56<00:00,  2.23it/s]


(Epoch 2) TRAIN LOSS:0.5188 F1:0.60 ARI:0.91 AMI:0.87 FM Score:0.92 Homogeneity Score:0.86 Completeness Score:0.90 V Measure:0.88 LR:0.00000300


(Epoch 3) TRAIN LOSS:0.4343 LR:0.00000300: 100%|██████████| 929/929 [07:01<00:00,  2.20it/s]


(Epoch 3) TRAIN LOSS:0.4343 F1:0.65 ARI:0.93 AMI:0.89 FM Score:0.93 Homogeneity Score:0.88 Completeness Score:0.91 V Measure:0.90 LR:0.00000300


(Epoch 4) TRAIN LOSS:0.3665 LR:0.00000300: 100%|██████████| 929/929 [07:00<00:00,  2.21it/s]


(Epoch 4) TRAIN LOSS:0.3665 F1:0.68 ARI:0.94 AMI:0.91 FM Score:0.94 Homogeneity Score:0.90 Completeness Score:0.92 V Measure:0.91 LR:0.00000300


(Epoch 5) TRAIN LOSS:0.3200 LR:0.00000300: 100%|██████████| 929/929 [06:59<00:00,  2.22it/s]

(Epoch 5) TRAIN LOSS:0.3200 F1:0.71 ARI:0.94 AMI:0.91 FM Score:0.94 Homogeneity Score:0.90 Completeness Score:0.92 V Measure:0.91 LR:0.00000300


In [ ]:
def predict(text):
  sentiments = []
  scores = []

  with torch.no_grad():
    for index in tqdm(text.index):
      subwords = tokenizer.encode(text[index], padding = True, truncation = True, max_length = 512, add_special_tokens = True)
      subwords = torch.LongTensor(subwords).view(1, -1).to(model.device)

      logits = model(subwords)[0]
      label = torch.topk(logits, k=1, dim=-1)[1].squeeze().item()

      sentiments.append(i2w[label])
      scores.append(F.softmax(logits, dim=-1).squeeze()[label])

    return sentiments, scores

sentimen, skor = predict(X_test['def_clean'])

100%|██████████| 3185/3185 [01:10<00:00, 45.48it/s]


In [ ]:
y_test = y_test.apply(lambda x: str(x))

In [ ]:
from sklearn.metrics import adjusted_rand_score, fowlkes_mallows_score, v_measure_score, homogeneity_score, completeness_score

print("ARI: ", adjusted_rand_score(sentimen, y_test))
print("AMI: ", adjusted_mutual_info_score(sentimen, y_test))
print("FM Score: ", fowlkes_mallows_score(sentimen, y_test))
print("Homogeneity: ", homogeneity_score(sentimen, y_test))
print("Completeness: ", completeness_score(sentimen, y_test))
print("V-Measure: ", v_measure_score(sentimen, y_test))
print("Accuracy score: ", accuracy_score(sentimen, y_test))
print("F1 Score: ", f1_score(sentimen, y_test, average = 'macro'))

ARI:  0.8902480778542795
AMI:  0.8776059680707337
FM Score:  0.8987045457398103
Homogeneity:  0.8978075371013784
Completeness:  0.8731540253643775
V-Measure:  0.8853091807719056
Accuracy score:  0.8973312401883831
F1 Score:  0.6941394598259341


**BP IEA**
- ARI:  0.8902480778542795
- AMI:  0.8776059680707337
- FM Score:  0.8987045457398103
- Homogeneity:  0.8978075371013784
- Completeness:  0.8731540253643775
- V-Measure:  0.8853091807719056
- Accuracy score:  0.8973312401883831
- F1 Score:  0.6941394598259341
- epoch: 10
- finetuning: 4200 second
- inference: 61 second

**BP NONIEA**
- ARI:  0.60334996454073
- AMI:  0.6207080027528805
- FM Score:  0.6454593414887023
- Homogeneity:  0.7068210002962748
- Completeness:  0.6218802668838864
- V-Measure:  0.6616356033681459
- Accuracy score:  0.737406216505895
- F1 Score:  0.3436237172606901
- epoch: 20
- finetuning: 1580 second
- inference: 13 second

**MF IEA**
- ARI:  0.9906987787909107
- AMI:  0.9899738141446977
- FM Score:  0.9913843586265855
- Homogeneity:  0.9928542830965557
- Completeness:  0.9884513787749896
- V-Measure:  0.9906479388166922
- Accuracy score:  0.9764309764309764
- F1 Score:  0.9593550854697518
- epoch: 20
- finetuning: 4920 second
- inference: 49 second

**MF NONIEA**
- ARI:  0.5793019190144
- AMI:  0.6939767330985349
- FM Score:  0.6241379290625336
- Homogeneity:  0.754286250915422
- Completeness:  0.714099523113524
- V-Measure:  0.7336429725710116
- Accuracy score:  0.7882926829268293
- F1 Score:  0.6822879942157624
- epoch: 50
- finetuning: 1500 second
- inference: 11 second

**CC IEA**
- ARI:  0.04728026721997087
- AMI:  0.1782048887555801
- FM Score:  0.2174578994905121
- Homogeneity:  0.30450961949225686
- Completeness:  0.17295318166404475
- V-Measure:  0.2206073746936897
- Accuracy score:  0.30828082808280827
- F1 Score:  0.15361172131729414
- epoch: 50
- finetuning: 2750 second
- inference: 25 second

**CC NONIEA**
- ARI:  0.055784236518897926
- AMI:  0.0759657814229736
- FM Score:  0.20338210825347694
- Homogeneity:  0.1841863448606275
- Completeness:  0.1058990067917829
- V-Measure:  0.13447870341775103
- Accuracy score:  0.2919621749408983
- F1 Score:  0.055001753386735214
- epoch: 50
- finetuning: 1250 second
- inference: 11 second